In [1]:
from google.colab import drive
drive.mount('/content/drive')

# === UPDATE THIS TO YOUR EXPERIMENT FOLDER ===
BASE_DIR    = "/content/drive/MyDrive/longformer_runs/run_paper_v1"
DATA_DIR    = f"{BASE_DIR}/data"
RESULTS_DIR = f"{BASE_DIR}/results_longformer"
SHAP_DIR    = f"{BASE_DIR}/shap_outputs"

import os
os.makedirs(SHAP_DIR, exist_ok=True)

print("DATA_DIR   :", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("SHAP_DIR   :", SHAP_DIR)


Mounted at /content/drive
DATA_DIR   : /content/drive/MyDrive/longformer_runs/run_paper_v1/data
RESULTS_DIR: /content/drive/MyDrive/longformer_runs/run_paper_v1/results_longformer
SHAP_DIR   : /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs


In [2]:
!pip install -q transformers pandas numpy matplotlib seaborn


In [3]:
import pandas as pd
import numpy as np
import os

test_path = os.path.join(DATA_DIR, "test.json")
test_df = pd.read_json(test_path)

def combine_example(code, comment):
    return (code or "") + "\n\n[COMMENT]\n" + (comment or "")

all_texts = [
    combine_example(c, m)
    for c, m in zip(test_df["new_code_raw"], test_df["new_comment_raw"])
]

all_labels = test_df["label"].astype(int).to_numpy()

print("Loaded test.json.")
print("Total samples:", len(all_texts))


Loaded test.json.
Total samples: 1066


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
import torch

# Load best checkpoint
best_ckpt_file = os.path.join(RESULTS_DIR, "BEST_CHECKPOINT.txt")
with open(best_ckpt_file) as f:
    best_ckpt = f.read().strip()

print("Using checkpoint:", best_ckpt)

tokenizer = AutoTokenizer.from_pretrained("allenai/longformer-base-4096")
model = AutoModelForSequenceClassification.from_pretrained(best_ckpt)

device = 0 if torch.cuda.is_available() else -1
clf = TextClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    device=device,
    return_all_scores=True,
)

pred_path = os.path.join(SHAP_DIR, "test_predictions.csv")

if os.path.isfile(pred_path):
    print("Loading cached predictions...")
    preds_df = pd.read_csv(pred_path)
else:
    print("Computing predictions...")
    all_probs = []
    all_pred = []

    BATCH = 16
    for i in range(0, len(all_texts), BATCH):
        batch = all_texts[i:i+BATCH]
        outs = clf(batch, truncation=True, max_length=1024)

        for row in outs:
            p0 = p1 = 0.0
            for d in row:
                lab = d["label"]
                idx = int(lab.replace("LABEL_","")) if lab.startswith("LABEL_") else int(lab)
                if idx == 0: p0 = d["score"]
                else: p1 = d["score"]
            all_probs.append((p0,p1))
            all_pred.append(1 if p1 >= p0 else 0)

    all_probs = np.array(all_probs)
    preds_df = pd.DataFrame({
        "label_test": all_labels,
        "p_class0": all_probs[:,0],
        "p_class1": all_probs[:,1],
        "pred_label": all_pred
    })

    preds_df.to_csv(pred_path, index=False)
    print("Saved:", pred_path)

preds_df.head()


Using checkpoint: /content/drive/MyDrive/longformer_runs/run_paper_v1/results_longformer/checkpoint-1575


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
/usr/local/lib/python3.12/dist-packages/transformers/pipelines/text_classification.py:111: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


Loading cached predictions...


,idx_in_test_after_cleaning,label_test,p_class0,p_class1,pred_label
0,0,1,0.947870,0.052130,0
1,1,0,0.949378,0.050622,0
2,2,1,0.824398,0.175602,0
3,3,0,0.945457,0.054543,0
4,4,1,0.469261,0.530739,1


In [5]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

y_true = preds_df["label_test"].to_numpy()
y_pred = preds_df["pred_label"].to_numpy()

cm = confusion_matrix(y_true, y_pred, labels=[0,1])

plt.figure(figsize=(6,5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred 0", "Pred 1"],
    yticklabels=["True 0", "True 1"]
)
plt.title("Confusion Matrix (Longformer Test Set)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()

cm_path = os.path.join(SHAP_DIR, "cm_test_full.png")
plt.savefig(cm_path, dpi=200, bbox_inches="tight")
plt.close()

print("Saved confusion matrix to:", cm_path)


Saved confusion matrix to: /content/drive/MyDrive/longformer_runs/run_paper_v1/shap_outputs/cm_test_full.png
